In [0]:
spark.conf.get("spark.sql.adaptive.enabled")

'false'

In [0]:
# partition coalescing
spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled")

'false'

In [0]:
spark.conf.set("spark.sql.adaptive.enabled","false")

spark.conf.set(
 "spark.sql.adaptive.coalescePartitions.enabled",
 "false"
)

spark.conf.set("spark.sql.shuffle.partitions", 100)

In [0]:
# source
ECOMM_BRONZE_FILES = "abfss://bronze@gopaldatalake.dfs.core.windows.net/ecomm-gopal/"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Step 1: Define explicit schema
ecomm_schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True), # the csv does not have sql datetime format
    StructField("UnitPrice", DoubleType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("Country", StringType(), True)
])

# LAZY , not yet started the stream
# Step 1: Read from Bronze Zone
bronze_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(ecomm_schema)
    .load(ECOMM_BRONZE_FILES)  
)

# .show will not work on stream
bronze_df.printSchema()
bronze_df.show(2)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
only showing top 2 rows


In [0]:
from pyspark.sql import functions as F

# Data Analysis per country 
country_counts = (
    bronze_df
      .groupBy("Country")
      .count()
      .orderBy(F.desc("count"))
)

country_counts.show(50, truncate=False)


+--------------------+------+
|Country             |count |
+--------------------+------+
|United Kingdom      |495478|
|Germany             |9495  |
|France              |8557  |
|EIRE                |8196  |
|Spain               |2533  |
|Netherlands         |2371  |
|Belgium             |2069  |
|Switzerland         |2002  |
|Portugal            |1519  |
|Australia           |1259  |
|Norway              |1086  |
|Italy               |803   |
|Channel Islands     |758   |
|Finland             |695   |
|Cyprus              |622   |
|Sweden              |462   |
|Unspecified         |446   |
|Austria             |401   |
|Denmark             |389   |
|Japan               |358   |
|Poland              |341   |
|Israel              |297   |
|USA                 |291   |
|Hong Kong           |288   |
|Singapore           |229   |
|Iceland             |182   |
|Canada              |151   |
|Greece              |146   |
|Malta               |127   |
|United Arab Emirates|68    |
|European 

In [0]:
bronze_df.selectExpr("spark_partition_id() as pid") \
  .distinct() \
  .count()

4

In [0]:
from pyspark.sql import functions as F

bronze_df.groupBy(F.spark_partition_id().alias("partition_id")) \
  .count() \
  .orderBy("partition_id") \
  .show(truncate=False)

+------------+------+
|partition_id|count |
+------------+------+
|0           |148766|
|1           |149224|
|2           |146863|
|3           |97056 |
+------------+------+



In [0]:
spark.conf.get("spark.sql.shuffle.partitions")

'100'

In [0]:
# data skew plus intential delay usign udf, as we have only 500k records.

import time
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

# 0.0005 seconds = 0.5 milliseconds = 500 microseconds, change it if needed
# 0.00001 - 10 microseconds
def slow(country):
    if country == "United Kingdom":
        time.sleep(0.00001)   # tune this
    return 1

 
slow_udf = udf(slow, IntegerType())

skew_df = bronze_df.repartition("Country")\
                  .withColumn("dummy", slow_udf("Country"))

skew_df.groupBy(F.spark_partition_id().alias("pid")) \
      .count() \
      .orderBy("pid") \
      .show(100)

+---+------+
|pid| count|
+---+------+
|  1|   288|
|  4|   479|
|  5|   758|
|  9|   462|
| 10|   291|
| 13|   622|
| 15|   229|
| 22|  9495|
| 25|    58|
| 30|  8557|
| 31|   146|
| 33|    10|
| 39|  2002|
| 44|   129|
| 45|  2069|
| 51|   151|
| 53|   695|
| 55|    30|
| 57|    32|
| 59|    45|
| 62|   127|
| 65|   358|
| 74|   341|
| 75|  1965|
| 76|  1259|
| 77|   803|
| 80|  8597|
| 83|    35|
| 84|  1086|
| 85|  2533|
| 86|    19|
| 90|   389|
| 95|495478|
| 97|  2371|
+---+------+



In [0]:
start = time.time()

(
skew_df
.groupBy("Country")
.agg(
    F.sum("Quantity"),
    F.avg("UnitPrice"),
    F.sum("dummy")   # forces UDF execution
)
.collect()
)

print("Elapsed:", time.time()-start)

Elapsed: 90.74772143363953


In [0]:
# Skew fix using salting, but you may rely on your own data, for example, if country is skewed, concat country with states, pluus distribute to distribute the data, salt is very common pattern, but you may align it with your own data rather than always using salt as solution

from pyspark.sql import functions as F

salt_buckets = 8

salted_df = bronze_df.withColumn(
    "salt",
    F.when(
        F.col("Country")=="United Kingdom",
        (F.rand()*salt_buckets).cast("int")
    ).otherwise(F.lit(0))
)

salted_df = salted_df.withColumn(
    "country_salted",
    F.concat(
       F.col("Country"),
       F.lit("_"),
       F.col("salt")
    )
)

# repartiton with salted key, we still use udf to cause slow, now we have salted key 
salted_df = salted_df.repartition("country_salted") \
                     .withColumn("dummy", slow_udf("Country"))

In [0]:
# over salted 
start=time.time()

(
salted_df
.groupBy("Country")
.agg(
   F.sum("Quantity"),
   F.avg("UnitPrice"),
   F.sum("dummy")
)
.collect()
)

print("Salted runtime:", time.time()-start)

Salted runtime: 38.383594274520874


In [0]:
skew_df.groupBy(
 F.spark_partition_id().alias("pid")
).count().show(100)

+---+------+
|pid| count|
+---+------+
|  4|   479|
| 31|   146|
| 55|    30|
| 59|    45|
| 85|  2533|
| 65|   358|
| 39|  2002|
| 53|   695|
| 84|  1086|
| 51|   151|
| 97|  2371|
| 76|  1259|
| 44|   129|
| 10|   291|
| 77|   803|
| 45|  2069|
| 22|  9495|
| 80|  8597|
| 25|    58|
| 62|   127|
| 95|495478|
|  1|   288|
| 13|   622|
| 86|    19|
| 90|   389|
| 75|  1965|
| 57|    32|
| 33|    10|
| 83|    35|
|  5|   758|
| 15|   229|
| 30|  8557|
|  9|   462|
| 74|   341|
+---+------+



In [0]:
salted_df.groupBy(
 F.spark_partition_id().alias("pid")
).count().show(100)

+---+-----+
|pid|count|
+---+-----+
| 65|  446|
| 49|62356|
| 84|11090|
| 28| 2069|
| 26|   19|
| 27| 9495|
| 63|  229|
| 10|  389|
| 77| 2371|
| 12|61779|
| 50|61779|
| 38|  758|
| 73| 9455|
| 52| 1272|
| 98|61894|
| 13|62053|
|  6|  462|
| 32|  622|
| 60|  182|
| 20| 1086|
| 94| 2002|
| 58|   32|
| 96|   35|
| 83|  151|
| 48|   10|
| 68|  341|
| 19|  695|
|  2|  588|
| 79|   45|
| 41|62065|
| 15|61745|
| 99|62156|
| 66|   58|
| 67|  358|
| 61| 1519|
| 17|  146|
| 35|  127|
| 36|   30|
+---+-----+

